# QDIST v4.0.0 — reviewed cohort standardization and event verification

This candidate reuses the immutable `qdist-v3.1.1` native-waveform extraction. It does **not** re-decode media or change detector thresholds. It standardizes the 519-recording evidence package, independently reconstructs all three features, evaluates morphology and episode grouping, builds the complete 60-item event review and A–J figure suite, and retains finalization/freeze as a separate reviewed decision.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.parent != PROJECT_ROOT and not (PROJECT_ROOT / "src reviewed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src reviewed").exists():
    raise FileNotFoundError("Could not resolve the Paper 1 project root.")

REVIEWED_SRC = PROJECT_ROOT / "src reviewed"
if str(REVIEWED_SRC) not in sys.path:
    sys.path.insert(0, str(REVIEWED_SRC))

from paper1_qc_reviewed.qdist_v400_cohort import (
    CohortPaths,
    run_cohort_review,
    verify_frozen_baseline,
    verify_preflight_bundle,
)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
RUN_PACKAGE_TESTS = True
RUN_COHORT_STANDARDIZATION = True
BUILD_EVENT_REVIEW = True
REBUILD_EVENT_REVIEW = False

# The immutable qdist-v3.1.1 feature extraction is reused; raw media are not re-decoded.
RECOMPUTE_FEATURE_EXTRACTION = False
PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"

if RECOMPUTE_FEATURE_EXTRACTION:
    raise ValueError("QDIST v4.0.0 cohort standardization must not recompute frozen v3.1.1 extraction.")
if PUBLISH_AND_FREEZE:
    raise ValueError("This notebook cannot publish or freeze QDIST.")
if SCIENTIFIC_REVIEW_DECISION != "PENDING":
    raise ValueError("Scientific review must remain PENDING in the cohort notebook.")

In [ ]:
paths = CohortPaths.from_project_root(PROJECT_ROOT)
preflight_manifest, preflight_figure_index = verify_preflight_bundle(paths.preflight_root)
frozen_manifest, frozen_data = verify_frozen_baseline(paths.frozen_root)

contract_summary = pd.DataFrame([
    {
        "contract": "accepted_preflight",
        "passed": bool(preflight_manifest["preflight_blocking_checks_pass"]),
        "observed": preflight_manifest["measurement_version"],
    },
    {
        "contract": "immutable_baseline",
        "passed": frozen_manifest["measurement_version"] == "qdist-v3.1.1",
        "observed": frozen_manifest["measurement_version"],
    },
    {
        "contract": "frozen_recordings",
        "passed": len(frozen_data["recordings"]) == 519,
        "observed": len(frozen_data["recordings"]),
    },
    {
        "contract": "preflight_panels",
        "passed": set(preflight_figure_index["panel"].astype(str)) == {"A", "B", "C"},
        "observed": sorted(preflight_figure_index["panel"].astype(str).unique()),
    },
])
display(contract_summary)
assert contract_summary["passed"].all()

In [ ]:
if RUN_PACKAGE_TESTS:
    os.environ["QDIST_PROJECT_ROOT"] = str(PROJECT_ROOT)
    test_files = [
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400.py",
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400_notebook.py",
        PROJECT_ROOT / "tests reviewed" / "test_qdist_v400_cohort.py",
    ]
    command = [sys.executable, "-m", "pytest", *map(str, test_files), "-q"]
    completed = subprocess.run(command, cwd=PROJECT_ROOT, text=True)
    if completed.returncode != 0:
        raise RuntimeError("QDIST reviewed package tests failed.")
else:
    print("Package tests skipped by explicit control.")

In [ ]:
if not RUN_COHORT_STANDARDIZATION:
    raise RuntimeError("RUN_COHORT_STANDARDIZATION must be True for the reviewed cohort run.")

result = run_cohort_review(
    PROJECT_ROOT,
    build_event_review=BUILD_EVENT_REVIEW,
    rebuild_event_review=REBUILD_EVENT_REVIEW,
    scientific_review_decision=SCIENTIFIC_REVIEW_DECISION,
    publish_and_freeze=PUBLISH_AND_FREEZE,
)

manifest = result["manifest"]
checks = result["checks"]
recording_table = result["recording_table"]
figure_index = result["figure_index"]
event_index = result["event_index"]
adjudication = result["adjudication"]
decisions = result["decisions"]

In [ ]:
display(checks)
display(adjudication)
display(decisions)

summary = pd.DataFrame([
    {"measure": "recordings", "value": manifest["recording_count"]},
    {"measure": "participants", "value": manifest["participant_count"]},
    {"measure": "positive recordings", "value": manifest["positive_recording_count"]},
    {"measure": "valid-zero recordings", "value": manifest["valid_zero_recording_count"]},
    {"measure": "accepted plateaus", "value": manifest["accepted_plateau_count"]},
    {"measure": "merged episodes", "value": manifest["episode_count"]},
    {"measure": "event-review items", "value": manifest["event_review_item_count"]},
    {"measure": "figure bundles", "value": manifest["figure_bundle_count"]},
])
display(summary)
print(json.dumps(manifest, indent=2))

In [ ]:
assert checks["passed"].astype(bool).all(), checks.loc[~checks["passed"].astype(bool)]
assert manifest["cohort_evidence_complete"] is True
assert manifest["numerical_equivalence_to_qdist_v311"] is True
assert manifest["recording_count"] == 519
assert manifest["participant_count"] == 224
assert manifest["event_review_item_count"] == 60
assert manifest["event_review_all_five_views"] is True
assert manifest["figure_bundle_count"] == 23
assert manifest["gallery_bundle_count"] >= 8
assert manifest["scientific_review_decision"] == "PENDING"
assert manifest["freeze_allowed"] is False
assert manifest["publish_and_freeze"] is False
assert manifest["feature_values_recomputed"] is False
assert manifest["family_scalar_constructed"] is False
assert manifest["standalone_gate_allowed"] is False

print("QDIST v4.0.0 REVIEWED COHORT STANDARDIZATION COMPLETE")
print("Candidate only. Final G10 decisions and immutable freezes remain pending post-cohort scientific review.")